### Data Profiling — E-commerce Sales Dataset

Quick automated overview to spot missing values, duplicates, inconsistent categories, and outliers **before** deciding which Excel formulas/fixes to apply.

### 0. Setup — install & import

In [20]:
import pandas as pd
from ydata_profiling import ProfileReport
pd.set_option('display.max_columns', None)

### 1. Load  data

In [21]:
RAW_FILE_PATH = "realistic_e_commerce_sales_data.csv"
CLEANED_FILE_PATH = "realistic_e_commerce_sales_data 11(Cleaned_Data).csv"                           

raw = pd.read_csv(RAW_FILE_PATH)
cleaned = pd.read_csv(CLEANED_FILE_PATH)

cleaned['Age'] = pd.to_numeric(cleaned['Age'], errors='coerce').astype('Int64')

print("Raw DataFrame:")
display(raw.head())
print("\nCleaned DataFrame:")
display(cleaned.head())

Raw DataFrame:


,Customer ID,Gender,Region,Age,Product Name,Category,Unit Price,Quantity,Total Price,Shipping Fee,Shipping Status,Order Date
0,CUST0268,Male,North,NaN,Monitor,Electronics,300.0,5,1500,13.31,Returned,2023-12-08
1,CUST0046,Male,West,22.0,Headphones,Accessories,100.0,2,200,6.93,In Transit,2023-04-09
2,CUST0169,Female,South,54.0,Monitor,Electronics,300.0,1,300,11.31,Returned,2023-08-28
3,CUST0002,Male,North,23.0,Headphones,Accessories,100.0,5,500,12.22,Delivered,2023-01-18
4,CUST0173,Female,South,NaN,Laptop,Electronics,1500.0,3,4500,5.40,Delivered,2023-01-19



Cleaned DataFrame:


,Customer ID,Gender,Division,Age,Product Name,Category,Unit Price,Quantity,Total Price,Shipping Fee,Shipping Status,Order Date,Month,Year,Day of Week,Day Type,Age Group,Quarter,Month Name,Customer Recency,Customer Frequency,Customer Monetary,Unnamed: 22
0,CUST0268,Male,Dhaka,21,Monitor,Electronics,36900,5,184500,13.31,Returned,12/8/2023,12,2023,Friday,Weekend,Young Adult (18-35),Q4,Dec,270,11,2028270,NaN
1,CUST0046,Female,Dhaka,22,Headphones,Accessories,12300,2,24600,6.93,In Transit,4/9/2023,4,2023,Sunday,Weekday,Young Adult (18-35),Q2,Apr,101,7,1421880,NaN
2,CUST0169,Female,Chattogram,54,Monitor,Electronics,36900,1,36900,11.31,Returned,8/28/2023,8,2023,Monday,Weekday,Middle Age (36-55),Q3,Aug,75,2,73800,NaN
3,CUST0002,Male,Rajshahi,23,Headphones,Accessories,12300,5,61500,12.22,Delivered,1/18/2023,1,2023,Wednesday,Weekday,Young Adult (18-35),Q1,Jan,43,7,387450,NaN
4,CUST0173,Male,Dhaka,65,Laptop,Electronics,184500,3,553500,5.40,Delivered,1/19/2023,1,2023,Thursday,Weekday,Senior (56+),Q1,Jan,15,12,2739210,NaN


### 2. Conflicting values per customer

In [22]:
# Customers with more than one distinct Gender value
gender_conflicts_raw = raw.groupby('Customer ID')['Gender'].nunique()
gender_conflicts_raw = gender_conflicts_raw[gender_conflicts_raw > 1]
print(f"Customers with conflicting Gender (Raw Data): {len(gender_conflicts_raw)}")
display(gender_conflicts_raw)

gender_conflicts_cleaned = cleaned.groupby('Customer ID')['Gender'].nunique()
gender_conflicts_cleaned = gender_conflicts_cleaned[gender_conflicts_cleaned > 1]
print(f"Customers with conflicting Gender (Cleaned): {len(gender_conflicts_cleaned)}")
display(gender_conflicts_cleaned)

Customers with conflicting Gender (Raw Data): 192


Customer ID
CUST0001    2
CUST0002    2
CUST0004    2
CUST0006    2
CUST0007    2
           ..
CUST0295    2
CUST0296    2
CUST0298    2
CUST0299    2
CUST0300    2
Name: Gender, Length: 192, dtype: int64

Customers with conflicting Gender (Cleaned): 0


Series([], Name: Gender, dtype: int64)

In [23]:
# Same check for Region
region_conflicts_raw = raw.groupby('Customer ID')['Region'].nunique() 
region_conflicts_raw = region_conflicts_raw[region_conflicts_raw > 1]
print(f"Customers with conflicting Region (Raw Data): {len(region_conflicts_raw)}")
display(region_conflicts_raw)

division_conflicts_cleaned = cleaned.groupby('Customer ID')['Division'].nunique()
division_conflicts_cleaned = division_conflicts_cleaned[division_conflicts_cleaned > 1]
print(f"Customers with conflicting Division (Cleaned): {len(division_conflicts_cleaned)}")
display(division_conflicts_cleaned)

Customers with conflicting Region (Raw Data): 217


Customer ID
CUST0002    3
CUST0003    2
CUST0004    2
CUST0006    4
CUST0007    2
           ..
CUST0295    3
CUST0296    4
CUST0297    2
CUST0298    3
CUST0299    4
Name: Region, Length: 217, dtype: int64

Customers with conflicting Division (Cleaned): 0


Series([], Name: Division, dtype: int64)

### 3. Accuracy check — does Unit Price × Quantity = Total Price?

In [24]:
expected_total_raw = (raw['Unit Price'] * raw['Quantity']).round(2)
mismatches = raw[expected_total_raw != raw['Total Price']]
print(f"Rows where Unit Price x Quantity != Total Price (Raw Data): {len(mismatches)}")
display(mismatches[['Customer ID', 'Unit Price', 'Quantity', 'Total Price']])

expected_total_cleaned = (cleaned['Unit Price'] * cleaned['Quantity']).round(2)
mismatches_cleaned = cleaned[expected_total_cleaned != cleaned['Total Price']]
print(f"Rows where Unit Price x Quantity != Total Price (Cleaned Data): {len(mismatches_cleaned)}")
display(mismatches_cleaned[['Customer ID', 'Unit Price', 'Quantity', 'Total Price']])

Rows where Unit Price x Quantity != Total Price (Raw Data): 20


,Customer ID,Unit Price,Quantity,Total Price
26,CUST0264,1658.432327,2,1600
83,CUST0219,62.191212,2,60
84,CUST0037,103.652020,4,200
110,CUST0223,62.191212,2,60
194,CUST0269,621.912122,5,1500
209,CUST0134,621.912122,5,1500
215,CUST0194,62.191212,1,30
226,CUST0264,103.652020,4,200
244,CUST0128,621.912122,3,900
339,CUST0219,207.304041,1,100


Rows where Unit Price x Quantity != Total Price (Cleaned Data): 0


,Customer ID,Unit Price,Quantity,Total Price


### 4. Consistency check — does each product's price is same?

In [25]:
price_check_raw = raw.groupby('Product Name')['Unit Price'].nunique()
inconsistent_raw = price_check_raw[price_check_raw > 1]
print("Products with inconsistent pricing (Raw):", len(inconsistent_raw))
print(inconsistent_raw)
print(raw.groupby('Product Name')['Unit Price'].unique())

price_check_cleaned = cleaned.groupby('Product Name')['Unit Price'].nunique()
inconsistent_cleaned = price_check_cleaned[price_check_cleaned > 1]
print("Products with inconsistent pricing (Cleaned):", len(inconsistent_cleaned))
print(inconsistent_cleaned)
print(cleaned.groupby('Product Name')['Unit Price'].unique())

Products with inconsistent pricing (Raw): 6
Product Name
Headphones    2
Keyboard      2
Laptop        2
Monitor       2
Mouse         2
Smartphone    2
Name: Unit Price, dtype: int64
Product Name
Headphones    [100.0, 207.30404083244343]
Keyboard       [50.0, 103.65202041622172]
Laptop        [1500.0, 3109.560612486652]
Monitor        [300.0, 621.9121224973303]
Mouse           [30.0, 62.19121224973304]
Smartphone    [800.0, 1658.4323266595477]
Smartwatch                        [200.0]
Name: Unit Price, dtype: object
Products with inconsistent pricing (Cleaned): 0
Series([], Name: Unit Price, dtype: int64)
Product Name
Headphones     [12300]
Keyboard        [6150]
Laptop        [184500]
Monitor        [36900]
Mouse           [3690]
Smartphone     [98400]
Smartwatch     [24600]
Name: Unit Price, dtype: object


### 4. Accuracy check — does each customer's RFM columns contains correct values?

In [26]:
# ---- 1. Recompute R/F/M independently ----
cleaned['Order Date'] = pd.to_datetime(cleaned['Order Date'])
reference_date = cleaned['Order Date'].max() + pd.Timedelta(days=1)

rfm_check = cleaned.groupby('Customer ID').agg(
    Expected_Frequency=('Customer ID', 'count'),
    Expected_Monetary=('Total Price', 'sum'),
    Last_Order=('Order Date', 'max')
).reset_index()
rfm_check['Expected_Recency'] = (reference_date - rfm_check['Last_Order']).dt.days

# ---- 2. Pull the Excel calculated values ----
excel_rfm = cleaned.groupby('Customer ID')[['Customer Recency', 'Customer Frequency', 'Customer Monetary']].first().reset_index()

# ---- 3. Merge/Join and Compare ----
comparison = rfm_check.merge(excel_rfm, on='Customer ID')

comparison['Recency_Match'] = comparison['Expected_Recency'] == comparison['Customer Recency']
comparison['Frequency_Match'] = comparison['Expected_Frequency'] == comparison['Customer Frequency']
comparison['Monetary_Match'] = comparison['Expected_Monetary'] == comparison['Customer Monetary']

mismatches = comparison[~(comparison['Recency_Match'] & comparison['Frequency_Match'] & comparison['Monetary_Match'])]
print(f"Customers with RFM mismatches: {len(mismatches)}")
mismatches[['Customer ID', 'Customer Recency', 'Expected_Recency', 'Customer Frequency', 'Expected_Frequency', 'Customer Monetary', 'Expected_Monetary']]

Customers with RFM mismatches: 0


,Customer ID,Customer Recency,Expected_Recency,Customer Frequency,Expected_Frequency,Customer Monetary,Expected_Monetary


### 5. Full automated report 

Generates one interactive HTML file covering nulls, duplicates, correlations, distributions, and cardinality warnings for the *entire* dataset.

In [27]:
profile = ProfileReport(cleaned, title="E-commerce Sales Data Profile", explorative=True)
profile.to_file("data_profile_report.html")
print("Saved: data_profile_report.html — open this file in your browser")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 70.27it/s]

Saved: data_profile_report.html — open this file in your browser
